# RNA Velocity Analysis with scVelo

Pipeline: velocyto loom → scVelo preprocessing → dynamical model → UMAP visualization

## 0. Parameters — change paths here

In [ ]:
# ─── INPUT ───────────────────────────────────────────────────────────────────
# Path to the merged loom file (or a list of per-sample loom files to merge)
LOOM_PATH = "path/to/merged.loom"   # <-- change this

# If you have separate per-sample loom files, set LOOM_PATH=None and fill this:
LOOM_LIST = []   # e.g. ["sample1.loom", "sample2.loom"]

# ─── OUTPUT ──────────────────────────────────────────────────────────────────
OUT_DIR = "results_velocity"

# ─── ANALYSIS SETTINGS ───────────────────────────────────────────────────────
# Column in adata.obs that stores cluster/cell-type labels (set to None to skip)
CLUSTER_KEY = "leiden"          # or "cell_type", "seurat_clusters", etc.

# scVelo model: "stochastic" (fast, good start) or "dynamical" (slower, more accurate)
VELOCITY_MODE = "dynamical"

# Number of highly variable genes
N_HVG = 2000

# Minimum counts thresholds for velocity gene filtering
MIN_SHARED_COUNTS = 20

# Random seed for reproducibility
SEED = 42

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import loompy

sc.settings.verbosity = 2
scv.settings.verbosity = 2
scv.settings.presenter_view = True   # larger fonts in plots
scv.set_figure_params("scvelo", dpi=100, dpi_save=200)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"scVelo version: {scv.__version__}")
print(f"Scanpy version: {sc.__version__}")

## 2. Load loom file(s)

In [ ]:
if LOOM_PATH:
    adata = scv.read(LOOM_PATH, cache=True)
elif LOOM_LIST:
    # Load and merge multiple per-sample loom files.
    # velocyto barcode format: "SampleID:ACGT...x"  →  we keep the prefix as batch label.
    adatas = []
    for path in LOOM_LIST:
        sample_name = os.path.splitext(os.path.basename(path))[0]
        ad = scv.read(path, cache=True)
        # Barcodes from velocyto look like "cellbarcode", make them unique per sample
        ad.obs_names = [f"{sample_name}:{bc}" for bc in ad.obs_names]
        ad.obs["sample"] = sample_name
        adatas.append(ad)
    adata = adatas[0].concatenate(
        adatas[1:],
        batch_key="batch",
        batch_categories=[os.path.splitext(os.path.basename(p))[0] for p in LOOM_LIST],
        index_unique=None,   # barcodes are already unique
    )
else:
    raise ValueError("Set either LOOM_PATH or LOOM_LIST.")

print(adata)
print("\nLayers:", list(adata.layers.keys()))

## 3. Inspect the loom structure

In [ ]:
# Show available obs / var columns from velocyto
print("obs columns:", adata.obs.columns.tolist())
print("var columns:", adata.var.columns.tolist())

# Proportion of spliced vs unspliced counts per cell
scv.pl.proportions(adata, groupby=None)

## 4. Basic QC and filtering

In [ ]:
# Use spliced layer as the main count matrix (standard for scVelo)
if adata.X is None or adata.X.sum() == 0:
    adata.X = adata.layers["spliced"].copy()

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# Mitochondrial gene fraction
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

# ── Adjust thresholds below based on the violins ──────────────────────────────
MT_THRESHOLD = 20   # % mitochondrial reads cutoff
MIN_COUNTS   = 500
MAX_COUNTS   = 50_000

adata = adata[
    (adata.obs.pct_counts_mt < MT_THRESHOLD) &
    (adata.obs.total_counts  > MIN_COUNTS)   &
    (adata.obs.total_counts  < MAX_COUNTS)
].copy()

print(f"After QC: {adata.n_obs} cells × {adata.n_vars} genes")

## 5. Scanpy embedding (normalize → HVG → PCA → UMAP)

In [ ]:
# Keep raw spliced counts before normalization
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat", subset=False)
print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

sc.tl.pca(adata, svd_solver="arpack", random_state=SEED)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

sc.pp.neighbors(adata, n_pcs=30, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(adata, resolution=0.5, random_state=SEED, key_added="leiden")

sc.pl.umap(adata, color=["leiden"], legend_loc="on data", title="Leiden clusters")

## 6. scVelo preprocessing

In [ ]:
# scVelo needs raw integer spliced/unspliced counts in adata.layers.
# The normalization above touched adata.X but not the layers, so layers are still raw.

scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=MIN_SHARED_COUNTS,
    n_top_genes=N_HVG,
    log=True,
)

scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

print(adata)

## 7. RNA velocity

In [ ]:
if VELOCITY_MODE == "dynamical":
    # Step 1: recover dynamics (rate parameters per gene)
    scv.tl.recover_dynamics(adata, n_jobs=-1)  # n_jobs=-1 → use all CPUs
    scv.tl.velocity(adata, mode="dynamical")
elif VELOCITY_MODE == "stochastic":
    scv.tl.velocity(adata, mode="stochastic")
else:
    raise ValueError("VELOCITY_MODE must be 'dynamical' or 'stochastic'")

scv.tl.velocity_graph(adata, n_jobs=-1)

## 8. Visualization

In [ ]:
# ── 8.1  Velocity stream on UMAP ─────────────────────────────────────────────
color_key = CLUSTER_KEY if CLUSTER_KEY and CLUSTER_KEY in adata.obs.columns else "leiden"

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color=color_key,
    legend_loc="right margin",
    title="RNA velocity stream",
    save=f"{OUT_DIR}/velocity_stream.png",
)

# ── 8.2  Velocity arrows on UMAP ─────────────────────────────────────────────
scv.pl.velocity_embedding(
    adata,
    basis="umap",
    arrow_length=3,
    arrow_size=2,
    color=color_key,
    save=f"{OUT_DIR}/velocity_arrows.png",
)

In [ ]:
# ── 8.3  Velocity confidence and length ──────────────────────────────────────
scv.tl.velocity_confidence(adata)

scv.pl.scatter(
    adata,
    color=["velocity_length", "velocity_confidence"],
    cmap="coolwarm",
    perc=[5, 95],
    save=f"{OUT_DIR}/velocity_confidence.png",
)

In [ ]:
# ── 8.4  PAGA velocity graph ──────────────────────────────────────────────────
scv.tl.paga(
    adata,
    groups=color_key,
    use_time_prior="velocity_pseudotime",
)

scv.pl.paga(
    adata,
    basis="umap",
    size=50,
    alpha=0.1,
    min_edge_width=2,
    node_size_scale=1.5,
    save=f"{OUT_DIR}/paga_velocity.png",
)

## 9. Velocity pseudotime

In [ ]:
scv.tl.velocity_pseudotime(adata)

scv.pl.scatter(
    adata,
    color="velocity_pseudotime",
    cmap="gnuplot",
    save=f"{OUT_DIR}/pseudotime.png",
)

## 10. Top velocity genes per cluster

In [ ]:
# Genes that best explain the velocity in each cluster
scv.tl.rank_velocity_genes(adata, groupby=color_key, min_corr=0.3)

df_top = scv.DataFrame(adata.uns["rank_velocity_genes"]["names"]).head(10)
print(df_top)
df_top.to_csv(f"{OUT_DIR}/top_velocity_genes_per_cluster.csv")

In [ ]:
# Phase portraits for the top gene in the first cluster
top_gene = df_top.iloc[0, 0]

scv.pl.velocity(
    adata,
    var_names=[top_gene],
    color=color_key,
    save=f"{OUT_DIR}/phase_portrait_{top_gene}.png",
)

## 11. (Dynamical mode only) Latent time

In [ ]:
if VELOCITY_MODE == "dynamical":
    scv.tl.latent_time(adata)

    scv.pl.scatter(
        adata,
        color="latent_time",
        color_map="gnuplot",
        size=80,
        save=f"{OUT_DIR}/latent_time.png",
    )

    # Top genes ordered along latent time
    top_genes = adata.var["fit_likelihood"].sort_values(ascending=False).index[:10]

    scv.pl.heatmap(
        adata,
        var_names=top_genes,
        sortby="latent_time",
        col_color=color_key,
        yticklabels=True,
        n_convolve=100,
        save=f"{OUT_DIR}/heatmap_latent_time.png",
    )
else:
    print("Latent time is only available in dynamical mode.")

## 12. Save results

In [ ]:
out_h5ad = f"{OUT_DIR}/adata_velocity.h5ad"
adata.write_h5ad(out_h5ad)
print(f"Saved: {out_h5ad}")
print(adata)